# FraudScope — 02 MLOps : Tracking, Registry & Serving
## Phase 2 · Dataset IEEE-CIS · MLflow · XGBoost · SHAP

**Objectifs de ce notebook** :
1. Installer et configurer MLflow comme serveur de tracking
2. Comparer 5 stratégies de resampling sur XGBoost avec tracking complet des runs
3. Identifier le meilleur modèle et l'enregistrer dans le MLflow Model Registry
4. Servir le modèle via une API REST
5. Expliquer les prédictions avec SHAP et logger les artefacts

**Prérequis** : avoir complété `01_exploration.ipynb` et suivi `docs/phase2_mlflow_guide.md`.

> ⚠️ Le serveur MLflow doit être lancé **avant** d'exécuter ce notebook :
> ```
> mlflow server --host 127.0.0.1 --port 8080 --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./mlartifacts
> ```

## 0. Imports et configuration

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
import mlflow
import mlflow.xgboost
from mlflow.tracking import MlflowClient

from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import (
    recall_score, f1_score, average_precision_score,
    precision_recall_curve
)

import xgboost as xgb
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

import shap

# ── Configuration ─────────────────────────────────────────────
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

FIGURES_DIR   = Path('figures')
ARTIFACTS_DIR = Path('artifacts')
FIGURES_DIR.mkdir(exist_ok=True)
ARTIFACTS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TARGET       = 'isFraud'

# ── MLflow setup ─────────────────────────────────────────────
load_dotenv()

TRACKING_URI        = os.getenv('MLFLOW_TRACKING_URI', 'http://127.0.0.1:8080')
EXPERIMENT_NAME     = os.getenv('MLFLOW_EXPERIMENT_NAME', 'fraud-detection-paytrack')
MODEL_REGISTRY_NAME = 'FraudScopeXGB'

mlflow.set_tracking_uri(TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow tracking URI : {TRACKING_URI}')
print(f'Expérience          : {EXPERIMENT_NAME}')
print(f'MLflow version      : {mlflow.__version__}')
print('Imports OK')

MLflow tracking URI : http://127.0.0.1:8080
Expérience          : fraud-detection-paytrack
MLflow version      : 3.14.0
Imports OK


## 1. Chargement et préparation des données

On réutilise les mêmes données que la Phase 1.
Les features de vélocité sont recalculées ici pour être disponibles dans ce notebook.

In [2]:
DATA_DIR = Path('data')

assert (DATA_DIR / 'train_transaction.csv').exists(), 'Données manquantes → voir data/get_dataset.md'

print('Chargement des données...')
train_tx = pd.read_csv(DATA_DIR / 'train_transaction.csv')
train_id = pd.read_csv(DATA_DIR / 'train_identity.csv')

df = train_tx.merge(train_id, on='TransactionID', how='left')
df = df.sort_values('TransactionDT').reset_index(drop=True)
print(f'Dataset : {df.shape[0]:,} transactions, {df.shape[1]} colonnes')

def reduce_mem_usage(df):
    for col in df.select_dtypes(include=['int64', 'float64']).columns:
        col_min, col_max = df[col].min(), df[col].max()
        if str(df[col].dtype).startswith('int'):
            if   col_min >= np.iinfo(np.int8).min  and col_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif col_min >= np.iinfo(np.int16).min and col_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            else:
                df[col] = df[col].astype(np.int32)
        else:
            if col_min >= np.finfo(np.float32).min and col_max <= np.finfo(np.float32).max:
                df[col] = df[col].astype(np.float32)
    return df

df = reduce_mem_usage(df)
print('Downcast mémoire appliqué.')

Chargement des données...
Dataset : 590,540 transactions, 434 colonnes
Downcast mémoire appliqué.


In [3]:
df['hour'] = (df['TransactionDT'] // 3600) % 24
df['day']  =  df['TransactionDT'] // (3600 * 24)

proxy_cols = [c for c in ['card1','card2','card3','card4','card5','card6','addr1','addr2'] if c in df.columns]
df['customer_proxy'] = df[proxy_cols].astype(str).fillna('NA').agg('_'.join, axis=1)
df['tx_time_sec']    = df['TransactionDT'].astype(float)

df['tx_rank_in_customer'] = df.groupby('customer_proxy').cumcount()

df['amount_ratio'] = df['TransactionAmt'] / (
    df.groupby('customer_proxy')['TransactionAmt']
    .transform('median')
    .clip(lower=0.01)
)
df['amount_ratio'] = df['amount_ratio'].replace([np.inf, -np.inf], 1.0).fillna(1.0)

print('Features temporelles et vélocité créées.')
print(df[['hour', 'day', 'tx_rank_in_customer', 'amount_ratio', TARGET]].describe())

Features temporelles et vélocité créées.
             hour         day  tx_rank_in_customer  amount_ratio     isFraud
count 590540.0000 590540.0000          590540.0000   590540.0000 590540.0000
mean      13.8619     84.7292             378.0832        1.5941      0.0350
std        7.6072     53.4373             977.5573        2.5241      0.1838
min        0.0000      1.0000               0.0000        0.0049      0.0000
25%        6.0000     35.0000               7.0000        0.6646      0.0000
50%       16.0000     84.0000              43.0000        1.0000      0.0000
75%       20.0000    130.0000             257.0000        1.6441      0.0000
max       23.0000    182.0000            9899.0000      319.3739      1.0000


## 2. Préparation du pipeline de prétraitement

In [4]:
exclude = {'TransactionID', TARGET, 'customer_proxy', 'tx_time_sec'}
high_missing = df.columns[df.isna().mean() > 0.95].tolist()
all_features = [c for c in df.columns if c not in exclude and c not in high_missing]

X = df[all_features]
y = df[TARGET]

num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print(f'Features numériques   : {len(num_features)}')
print(f'Features catégorielles : {len(cat_features)}')
print(f'Total                 : {len(all_features)}')

num_transformer = SimpleImputer(strategy='median')

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
], remainder='drop')

split_idx = int(len(df) * 0.8)
X_train_raw, X_test_raw = X.iloc[:split_idx], X.iloc[split_idx:]
y_train,     y_test     = y.iloc[:split_idx], y.iloc[split_idx:]

X_train = preprocessor.fit_transform(X_train_raw)
X_test  = preprocessor.transform(X_test_raw)

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Ratio fraude train : {y_train.mean():.4f}  |  test : {y_test.mean():.4f}')

Features numériques   : 398
Features catégorielles : 29
Total                 : 427
Train : (472432, 427)  |  Test : (118108, 427)
Ratio fraude train : 0.0351  |  test : 0.0344


## 3. Comparaison des stratégies de resampling avec MLflow

On compare 5 stratégies pour gérer le déséquilibre de classes :

| # | Stratégie | Principe |
|---|-----------|----------|
| 1 | **Baseline** | Aucun resampling — XGBoost sur données brutes |
| 2 | **Class weighting** | `scale_pos_weight` = nb_négatifs / nb_positifs |
| 3 | **Undersampling** | Réduction aléatoire de la classe majoritaire |
| 4 | **SMOTE** | Génération synthétique d'exemples frauduleux |
| 5 | **SMOTE+ENN** | SMOTE + nettoyage des frontières de décision |

**Chaque stratégie génère un run MLflow indépendant**, loggant paramètres, métriques et artefacts.

In [5]:
XGB_BASE_PARAMS = {
    'n_estimators':     300,
    'max_depth':        6,
    'learning_rate':    0.05,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'eval_metric':      'aucpr',
    'early_stopping_rounds': 20,
    'random_state':     RANDOM_STATE,
    'n_jobs':           -1
}

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
imbalance_ratio = neg_count / pos_count
print(f'Ratio déséquilibre : {imbalance_ratio:.1f}  ({neg_count:,} légitimes / {pos_count:,} fraudes)')

Ratio déséquilibre : 27.5  (455,833 légitimes / 16,599 fraudes)


In [6]:
def run_strategy(strategy_name, X_tr, y_tr, X_val, y_val, extra_params=None):
    """Entraîne un XGBoost et logue tout dans MLflow."""
    params = {**XGB_BASE_PARAMS, **(extra_params or {})}

    with mlflow.start_run(run_name=f'xgb-{strategy_name}') as run:

        mlflow.set_tags({'dataset': 'IEEE-CIS', 'model': 'XGBoost', 'strategy': strategy_name, 'phase': '2'})
        mlflow.log_params(params)
        mlflow.log_param('train_size', len(y_tr))
        mlflow.log_param('fraud_rate_train', float(y_tr.mean()))

        t0 = time.time()
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        fit_time = time.time() - t0

        y_proba = model.predict_proba(X_val)[:, 1]
        y_pred  = (y_proba >= 0.5).astype(int)

        auprc  = average_precision_score(y_val, y_proba)
        recall = recall_score(y_val, y_pred)
        f1     = f1_score(y_val, y_pred)

        mlflow.log_metrics({'AUPRC': auprc, 'recall_fraud': recall, 'f1': f1, 'fit_time_sec': fit_time})

        precision_vals, recall_vals, _ = precision_recall_curve(y_val, y_proba)
        fig_pr, ax_pr = plt.subplots(figsize=(7, 5))
        ax_pr.plot(recall_vals, precision_vals, lw=2, color='steelblue')
        ax_pr.fill_between(recall_vals, precision_vals, alpha=0.1, color='steelblue')
        ax_pr.set(xlabel='Recall', ylabel='Precision',
                  title=f'Courbe PR — {strategy_name} (AUPRC={auprc:.4f})',
                  xlim=(0,1), ylim=(0,1))
        mlflow.log_figure(fig_pr, 'pr_curve.png')
        plt.close(fig_pr)

        # Fix : utiliser name= au lieu du paramètre artifact_path= déprécié (mlflow >= 3.x)
        mlflow.xgboost.log_model(model, name='model')

        print(f'  [{strategy_name:20s}] AUPRC={auprc:.4f}  Recall={recall:.4f}  F1={f1:.4f}  ({fit_time:.1f}s)')
        return {'AUPRC': auprc, 'recall_fraud': recall, 'f1': f1, 'fit_time_sec': fit_time,
                'run_id': run.info.run_id, 'strategy': strategy_name}


In [7]:
all_results = []
print('=== Entraînement des 5 stratégies — runs MLflow loggés ===\n')

# 1 — Baseline
all_results.append(run_strategy('baseline', X_train, y_train, X_test, y_test))

# 2 — Class weighting
all_results.append(run_strategy('class_weighting', X_train, y_train, X_test, y_test,
                                extra_params={'scale_pos_weight': imbalance_ratio}))

# 3 — Undersampling
from imblearn.under_sampling import RandomUnderSampler
X_under, y_under = RandomUnderSampler(sampling_strategy=0.1, random_state=RANDOM_STATE).fit_resample(X_train, y_train)
all_results.append(run_strategy('undersampling', X_under, y_under, X_test, y_test))

=== Entraînement des 5 stratégies — runs MLflow loggés ===

  [baseline            ] AUPRC=0.5296  Recall=0.3484  F1=0.4844  (105.9s)
🏃 View run xgb-baseline at: http://127.0.0.1:8080/#/experiments/1/runs/4a1f686da0974429a8973bf0fbcc600b
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
  [class_weighting     ] AUPRC=0.5048  Recall=0.7320  F1=0.3277  (91.0s)
🏃 View run xgb-class_weighting at: http://127.0.0.1:8080/#/experiments/1/runs/bdf2262385ef4e12a4e904dfd3e44252
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
  [undersampling       ] AUPRC=0.5202  Recall=0.4235  F1=0.5019  (59.0s)
🏃 View run xgb-undersampling at: http://127.0.0.1:8080/#/experiments/1/runs/855aca4520944ad089e4e75a5c4ced1e
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1


In [8]:
# 4 — SMOTE  (n_jobs supprimé : non supporté dans imbalanced-learn >= 0.11)
X_smote, y_smote = SMOTE(sampling_strategy=0.2, random_state=RANDOM_STATE).fit_resample(X_train, y_train)
print(f'  SMOTE : {y_smote.sum():,} fraudes synthétiques + réelles dans le train')
all_results.append(run_strategy('smote', X_smote, y_smote, X_test, y_test))

# 5 — SMOTE+ENN
# Fix MemoryError : sampling_strategy 0.2 → 0.1 pour limiter la taille du tableau resamplé
# (0.2 générait ~547k×427 en float64 ≈ 1.74 GiB non allouable).
# Cast préalable en float32 pour réduire l'empreinte mémoire de moitié.
X_train_f32 = X_train.astype(np.float32)
print('  SMOTE+ENN : resampling sur sous-ensemble float32 (sampling_strategy=0.1)')
X_smoteenn, y_smoteenn = SMOTEENN(sampling_strategy=0.1, random_state=RANDOM_STATE).fit_resample(X_train_f32, y_train)
all_results.append(run_strategy('smote_enn', X_smoteenn, y_smoteenn, X_test, y_test))

results_df = pd.DataFrame(all_results).set_index('strategy')
print('\n=== Résultats comparatifs ===')
print(results_df[['AUPRC', 'recall_fraud', 'f1', 'fit_time_sec']].to_string())

  SMOTE : 91,166 fraudes synthétiques + réelles dans le train
  [smote               ] AUPRC=0.4910  Recall=0.3307  F1=0.4622  (167.3s)
🏃 View run xgb-smote at: http://127.0.0.1:8080/#/experiments/1/runs/d3971d4f794c44b08dd2e1837554c0bd
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1
  SMOTE+ENN : resampling sur sous-ensemble float32 (sampling_strategy=0.1)
  [smote_enn           ] AUPRC=0.5101  Recall=0.3890  F1=0.4751  (195.2s)
🏃 View run xgb-smote_enn at: http://127.0.0.1:8080/#/experiments/1/runs/e1c832a9b0514f3da7bd29f1c0c67a44
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/1

=== Résultats comparatifs ===
                  AUPRC  recall_fraud       f1  fit_time_sec
strategy                                                    
baseline         0.5296        0.3484   0.4844       105.900
class_weighting  0.5048        0.7320   0.3277        91.000
undersampling    0.5202        0.4235   0.5019        59.000
smote            0.4910        0.3307   0.4622       

## 4. Sélection du meilleur modèle

On interroge MLflow pour récupérer le run avec le meilleur AUPRC.
C'est **MLflow qui fait autorité** : pas de variable Python en dur.

In [ ]:
df_runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=['metrics.AUPRC DESC']
)

print('=== Runs de l experience (tries par AUPRC decroissant) ===')
print(df_runs[['tags.mlflow.runName', 'metrics.AUPRC', 'metrics.recall_fraud', 'metrics.f1', 'run_id']].to_string())

best_run    = df_runs.iloc[0]
best_run_id = best_run['run_id']
best_auprc  = best_run['metrics.AUPRC']
best_name   = best_run['tags.mlflow.runName']

print(f'\n✅ Meilleur run : {best_name}')
print(f'   AUPRC       : {best_auprc:.4f}')
print(f'   Run ID      : {best_run_id}')

## 5. Enregistrement dans le Model Registry

In [ ]:
model_uri = f'runs:/{best_run_id}/model'

print(f"Enregistrement du modèle sous '{MODEL_REGISTRY_NAME}'...")
registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)
model_version = registered_model.version
print(f'✅ Modèle enregistré : {MODEL_REGISTRY_NAME} v{model_version}')

client = MlflowClient()
client.update_model_version(
    name=MODEL_REGISTRY_NAME, version=model_version,
    description=f'XGBoost entraine sur IEEE-CIS. Strategie : {best_name}. AUPRC : {best_auprc:.4f}'
)

In [ ]:
print(f'Transition {MODEL_REGISTRY_NAME} v{model_version} : None → Staging')
client.transition_model_version_stage(name=MODEL_REGISTRY_NAME, version=model_version, stage='Staging')

AUPRC_THRESHOLD = 0.50

if best_auprc >= AUPRC_THRESHOLD:
    print(f'✅ AUPRC={best_auprc:.4f} >= seuil={AUPRC_THRESHOLD} -> Promotion en Production')
    client.transition_model_version_stage(name=MODEL_REGISTRY_NAME, version=model_version, stage='Production')
else:
    print(f'❌ AUPRC={best_auprc:.4f} < seuil={AUPRC_THRESHOLD} -> Reste en Staging.')

print(f"\nModele visible dans l'UI MLflow -> onglet 'Models' : http://127.0.0.1:8080")

## 6. Serving : tester l'API REST

Avant d'exécuter cette section, lance le serveur de prédiction dans un nouveau terminal :

```
mlflow models serve --model-uri models:/FraudScopeXGB/Production --host 127.0.0.1 --port 5001 --no-conda
```

In [ ]:
import requests, json

SERVING_URL    = 'http://127.0.0.1:5001/invocations'
sample_indices = [0, 1, 2]

X_sample = X_test[sample_indices].toarray() if hasattr(X_test, 'toarray') else X_test[sample_indices]

payload = {
    'dataframe_split': {
        'columns': [str(i) for i in range(X_sample.shape[1])],
        'data':    X_sample.tolist()
    }
}

try:
    response = requests.post(SERVING_URL, headers={'Content-Type': 'application/json'},
                             data=json.dumps(payload), timeout=5)
    response.raise_for_status()
    predictions = response.json()['predictions']
    print('Predictions API REST (probabilite de fraude) :')
    for i, (proba, label) in enumerate(zip(predictions, y_test.iloc[sample_indices].values)):
        flag = '🚨' if proba > 0.5 else '✅'
        print(f'  Transaction {i+1} : P(fraude)={proba:.4f}  (vraie etiquette={label})  {flag}')
except requests.exceptions.ConnectionError:
    print('⚠️  Serveur de prediction non demarre.')
    print('   Lance : mlflow models serve --model-uri models:/FraudScopeXGB/Production --host 127.0.0.1 --port 5001 --no-conda')

## 7. Explainabilite SHAP

SHAP (SHapley Additive exPlanations) permet de comprendre **pourquoi** le modele a predit fraude ou legitime.

- **Beeswarm global** : quelles features influencent le plus le modele sur l'ensemble du test set
- **Force plot local** : pourquoi le modele a dit "fraude" sur une transaction precise

In [ ]:
print('Chargement du modele depuis MLflow Registry...')
loaded_model = mlflow.xgboost.load_model(f'models:/{MODEL_REGISTRY_NAME}/Production')

N_SHAP = 500
X_shap = X_test[:N_SHAP].toarray() if hasattr(X_test, 'toarray') else X_test[:N_SHAP]

print(f'Calcul des valeurs SHAP sur {N_SHAP} transactions...')
explainer   = shap.TreeExplainer(loaded_model)
shap_values = explainer.shap_values(X_shap)

cat_feature_names = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(cat_features).tolist()
feature_names = num_features + cat_feature_names
print('Valeurs SHAP calculees.')

In [ ]:
shap.summary_plot(shap_values, features=X_shap, feature_names=feature_names, max_display=20, plot_type='dot', show=False)
plt.title('SHAP Beeswarm - Top 20 features influentes (FraudScopeXGB)')
plt.tight_layout()

with mlflow.start_run(run_id=best_run_id):
    mlflow.log_figure(plt.gcf(), 'shap_beeswarm.png')

plt.savefig(FIGURES_DIR / 'shap_beeswarm_phase2.png', dpi=150)
plt.show()
print('Beeswarm logge dans MLflow et sauvegarde dans figures/')

In [ ]:
y_test_vals  = y_test.values[:N_SHAP]
y_proba_shap = loaded_model.predict_proba(X_shap)[:, 1]
tp_indices   = np.where((y_test_vals == 1) & (y_proba_shap >= 0.5))[0]

if len(tp_indices) > 0:
    idx_tp = tp_indices[0]
    print(f'Vrai positif (index {idx_tp}) : P(fraude)={y_proba_shap[idx_tp]:.4f}')
    shap.force_plot(explainer.expected_value, shap_values[idx_tp],
                   features=X_shap[idx_tp], feature_names=feature_names,
                   matplotlib=True, show=False)
    plt.title(f'SHAP Force Plot - Vrai Positif (index {idx_tp})')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'shap_force_vrai_positif.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Pas de vrai positif dans les 500 premieres transactions. Augmenter N_SHAP.')

## 8. Bilan Phase 2

| Element | Statut |
|---------|--------|
| Serveur MLflow local | ✅ |
| 5 strategies XGBoost logguees | ✅ |
| Courbes PR artefacts MLflow | ✅ |
| Meilleur modele dans Registry | ✅ |
| Validation gate AUPRC | ✅ |
| Serving REST `/invocations` | ✅ |
| SHAP beeswarm global | ✅ |
| SHAP force plot local | ✅ |

**Prochaine etape** : Phase 3 - Features graphe (NetworkX) + GNN sur dataset Elliptic (GCN/GAT via PyTorch Geometric).

Le notebook `03_graphnn.ipynb` reprendra l'entrainement dans Jupyter, sur un dataset different (graphe de transactions Bitcoin).